# Waktu Intervensi Nonfarmasi

**ID proyek:** `O005-LEGA-V101-PRJ03`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Seberapa besar perubahan puncak epidemi ketika intervensi nonfarmasi yang sama dimulai pada waktu berbeda?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082203
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Model SIR tertutup memakai satu penurunan laju transmisi pada hari intervensi; kepatuhan langsung dan tetap; kondisi awal sama pada semua skenario.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
gamma, beta0, reduction = 0.10, 0.42, 0.42
t_eval = np.linspace(0.0, 150.0, 601)

def run_sir(start_day):
    def rhs(t, y):
        S, I, R = y
        beta_t = beta0 if start_day is None or t < start_day else beta0 * reduction
        return [-beta_t * S * I, beta_t * S * I - gamma * I, gamma * I]
    return solve_ivp(rhs, (0.0, 150.0), [0.999, 0.001, 0.0], t_eval=t_eval, rtol=1e-8, atol=1e-10, max_step=0.25)

npi_runs = {"hari 12": run_sir(12.0), "hari 22": run_sir(22.0), "tanpa NPI": run_sir(None)}
peaks = {name: float(run.y[1].max()) for name, run in npi_runs.items()}


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
for run in npi_runs.values():
    assert run.success and np.min(run.y) > -1e-9
    np.testing.assert_allclose(run.y.sum(axis=0), 1.0, atol=2e-8)
assert peaks["hari 12"] < peaks["hari 22"] < peaks["tanpa NPI"]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for name, run in npi_runs.items():
    ax.plot(t_eval, run.y[1], label=f"{name}; puncak={peaks[name]:.3f}")
ax.axvline(12, color="gray", linestyle=":", linewidth=1)
ax.axvline(22, color="gray", linestyle=":", linewidth=1)
ax.set(xlabel="hari", ylabel="fraksi terinfeksi", title="Pengaruh waktu NPI pada puncak epidemi")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Intervensi tidak memiliki biaya, penundaan, kelelahan, heterogenitas, atau respons perilaku endogen.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
